In [6]:
!pip install cvxpy --quiet
import cvxpy as cp
import numpy as np

#Dados do sistema
n_usinas = 3
n_linhas = 3
n_cargas = 3

#Custo incremental das usinas
c = np.array([10, 20, 30]) # $/MWh

#Capacidade das usinas
P_max = np.array([100, 150, 200]) # MW

#Capacidade das linhas de transmissão
F_max = np.array([50, 70, 80]) # MW

#Demanda das cargas
D = np.array([50, 70, 100]) # MW

#Matriz de incidência das linhas
A = np.array([
    [1, -1, 0],
    [1, 0, -1],
    [0, 1, -1]
])

#Variáveis de otimização
P = cp.Variable(n_usinas) # Geração das usinas
F = cp.Variable(n_linhas) # Fluxo nas linhas de transmissão
theta = cp.Variable(n_cargas) # Ângulo das tensões

#Função objetivo
obj = cp.Minimize(c @ P)

#Restrições
constraints = [
    cp.sum(P) == cp.sum(D), # Equação de balanço de potência
    P <= P_max, # Limite de geração das usinas
    P >= 0,       # Restrição de geração positiva
    F == A @ theta, # Equação de fluxo de potência
    F <= F_max, # Limite de fluxo nas linhas de transmissão
    theta[0] == 0 # Ângulo de referência
]

#Problema de otimização
prob = cp.Problem(obj, constraints)
prob.solve()

#Solução
P_opt = P.value
F_opt = F.value
theta_opt = theta.value

print("Geração das usinas (MW):")
for i in range(n_usinas):
    print(f"  Usina {i+1}: {P_opt[i]:.1f} MW")

print("\nFluxo nas linhas de transmissão (MW):")
# Assuming lines connect as: L1: Barra 1-2, L2: Barra 1-3, L3: Barra 2-3 based on matrix A
print(f"  Linha 1 (Barra 1-2): {F_opt[0]:.1f} MW")
print(f"  Linha 2 (Barra 1-3): {F_opt[1]:.1f} MW")
print(f"  Linha 3 (Barra 2-3): {F_opt[2]:.1f} MW")

print("\nÂngulo das tensões (rad):")
for i in range(n_cargas):
    print(f"  Barra {i+1}: {theta_opt[i]:.1f} rad")

print(f"\nCusto total da geração: {prob.value:.2f} $/hora")

Geração das usinas (MW):
  Usina 1: 100.0 MW
  Usina 2: 120.0 MW
  Usina 3: -0.0 MW

Fluxo nas linhas de transmissão (MW):
  Linha 1 (Barra 1-2): 14.7 MW
  Linha 2 (Barra 1-3): 59.4 MW
  Linha 3 (Barra 2-3): 44.7 MW

Ângulo das tensões (rad):
  Barra 1: 0.0 rad
  Barra 2: -14.7 rad
  Barra 3: -59.4 rad

Custo total da geração: 3400.00 $/hora
